<a href="https://colab.research.google.com/github/asahedev/ds-ml-guides/blob/main/data-science/pandas_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Pandas — Basics Reference Guide

A practical, runnable guide to the most essential Pandas concepts for data analysis.

**Topics covered:**
1. [Series & DataFrame](#1)
2. [Loading & Exploring Data](#2)
3. [Selecting Data — loc vs iloc](#3)
4. [Filtering Data](#4)
5. [Handling Missing Data](#5)
6. [GroupBy](#6)
7. [Merging DataFrames](#7)
8. [Creating New Columns & apply()](#8)
9. [Sorting](#9)
10. [Duplicates](#10)
11. [Pivot Tables](#11)
12. [Quick Reference Cheatsheet](#12)

---

In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")

Pandas version: 2.2.2


<a id='1'></a>
---
## 1. Series & DataFrame

Pandas has two core data structures:

| Structure | Dimensions | Analogy |
|-----------|-----------|--------|
| **Series** | 1D | A single spreadsheet column |
| **DataFrame** | 2D | A full spreadsheet table |

> 💡 **Key idea:** A DataFrame is simply a collection of Series that share the same index.

In [ ]:
# --- Series ---
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print("Series:")
print(s)
print("\nAccess by label:", s["b"])
print("Data type:", s.dtype)

Series:
a    10
b    20
c    30
dtype: int64

Access by label: 20
Data type: int64


In [ ]:
# --- DataFrame ---
df = pd.DataFrame({
    "emp_id":    [1, 2, 3, 4, 5],
    "name":      ["Alice", "Beth", "Carol", "Dana", "Eve"],
    "age":       [25, 30, 22, 35, 28],
    "salary":    [70000, 90000, 65000, 120000, None],
    "dept_id":   [10, 20, 10, 30, 20],
    "years_exp": [3.0, 7.0, None, 9.0, 5.0]
})

df

,emp_id,name,age,salary,dept_id,years_exp
0,1,Alice,25,70000.0,10,3.0
1,2,Beth,30,90000.0,20,7.0
2,3,Carol,22,65000.0,10,NaN
3,4,Dana,35,120000.0,30,9.0
4,5,Eve,28,NaN,20,5.0


In [ ]:
# A single column IS a Series
print(type(df["salary"]))
print(df["salary"])

<class 'pandas.core.series.Series'>
0     70000.0
1     90000.0
2     65000.0
3    120000.0
4         NaN
Name: salary, dtype: float64


<a id='2'></a>
---
## 2. Loading & Exploring Data

In real projects, data usually comes from external files. Here are the most common loading functions and the first commands you should run on any new dataset.

> 💡 **Tip:** Always run `head()`, `shape`, and `info()` as your first three steps when receiving new data.

In [ ]:
# --- Loading data (common formats) ---

# df = pd.read_csv("file.csv")          # CSV
# df = pd.read_excel("file.xlsx")       # Excel
# df = pd.read_json("file.json")        # JSON
# df = pd.read_sql(query, connection)   # SQL database

# We'll use the df created above — let's explore it
print("--- head() — first 5 rows ---")
display(df.head())

--- head() — first 5 rows ---


,emp_id,name,age,salary,dept_id,years_exp
0,1,Alice,25,70000.0,10,3.0
1,2,Beth,30,90000.0,20,7.0
2,3,Carol,22,65000.0,10,NaN
3,4,Dana,35,120000.0,30,9.0
4,5,Eve,28,NaN,20,5.0


In [ ]:
print("--- shape — (rows, columns) ---")
df.shape

--- shape — (rows, columns) ---


(5, 6)

In [ ]:
print("--- info() — column names, types, null counts ---")
df.info()

--- info() — column names, types, null counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   emp_id     5 non-null      int64  
 1   name       5 non-null      object 
 2   age        5 non-null      int64  
 3   salary     4 non-null      float64
 4   dept_id    5 non-null      int64  
 5   years_exp  4 non-null      float64
dtypes: float64(2), int64(3), object(1)
memory usage: 372.0+ bytes


In [ ]:
print("--- describe() — summary statistics ---")
df.describe()

--- describe() — summary statistics ---


,emp_id,age,salary,dept_id,years_exp
count,5.000000,5.000000,4.000000,5.0000,4.000000
mean,3.000000,28.000000,86250.000000,18.0000,6.000000
std,1.581139,4.949747,24958.298553,8.3666,2.581989
min,1.000000,22.000000,65000.000000,10.0000,3.000000
25%,2.000000,25.000000,68750.000000,10.0000,4.500000
50%,3.000000,28.000000,80000.000000,20.0000,6.000000
75%,4.000000,30.000000,97500.000000,20.0000,7.500000
max,5.000000,35.000000,120000.000000,30.0000,9.000000


In [ ]:
# Other useful exploration commands
print("Column names:", df.columns.tolist())
print("Data types:\n", df.dtypes)
print("\nTail (last 3 rows):")
display(df.tail(3))

Column names: ['emp_id', 'name', 'age', 'salary', 'dept_id', 'years_exp']
Data types:
 emp_id         int64
name          object
age            int64
salary       float64
dept_id        int64
years_exp    float64
dtype: object

Tail (last 3 rows):


,emp_id,name,age,salary,dept_id,years_exp
2,3,Carol,22,65000.0,10,NaN
3,4,Dana,35,120000.0,30,9.0
4,5,Eve,28,NaN,20,5.0


<a id='3'></a>
---
## 3. Selecting Data — `loc` vs `iloc`

| Method | Selects by | Slice end |
|--------|-----------|----------|
| `loc` | Label / name | **Inclusive** |
| `iloc` | Integer position | **Exclusive** |

> ⚠️ **Common gotcha:** `df.loc[0:2]` returns rows 0, 1, AND 2. `df.iloc[0:2]` returns only rows 0 and 1.

In [ ]:
# --- Selecting columns ---
print("Single column (returns Series):")
print(df["name"])

print("\nMultiple columns (returns DataFrame):")
display(df[["name", "salary"]])

Single column (returns Series):
0    Alice
1     Beth
2    Carol
3     Dana
4      Eve
Name: name, dtype: object

Multiple columns (returns DataFrame):


,name,salary
0,Alice,70000.0
1,Beth,90000.0
2,Carol,65000.0
3,Dana,120000.0
4,Eve,NaN


In [ ]:
# --- loc: select by label ---
print("Single value (row 1, column 'name'):")
print(df.loc[1, "name"])

print("\nRows 0-2, columns 'name' through 'salary':")
display(df.loc[0:2, "name":"salary"])

Single value (row 1, column 'name'):
Beth

Rows 0-2, columns 'name' through 'salary':


,name,age,salary
0,Alice,25,70000.0
1,Beth,30,90000.0
2,Carol,22,65000.0


In [ ]:
# --- iloc: select by position ---
print("Single value (row 0, column position 1):")
print(df.iloc[0, 1])

print("\nFirst 2 rows, first 3 columns (exclusive end):")
display(df.iloc[0:2, 0:3])

Single value (row 0, column position 1):
Alice

First 2 rows, first 3 columns (exclusive end):


,emp_id,name,age
0,1,Alice,25
1,2,Beth,30


<a id='4'></a>
---
## 4. Filtering Data

> 💡 **Key idea:** Filtering works by creating a boolean mask (a Series of True/False values) and applying it to the DataFrame.

> ⚠️ **Always wrap each condition in parentheses** when combining with `&` or `|`.

In [ ]:
# --- Single condition ---
print("Employees with salary > 70000:")
display(df[df["salary"] > 70000])

Employees with salary > 70000:


,emp_id,name,age,salary,dept_id,years_exp
1,2,Beth,30,90000.0,20,7.0
3,4,Dana,35,120000.0,30,9.0


In [ ]:
# --- Multiple conditions ---
# & = AND,  | = OR
print("Age > 24 AND dept_id == 10:")
display(df[(df["age"] > 24) & (df["dept_id"] == 10)])

Age > 24 AND dept_id == 10:


,emp_id,name,age,salary,dept_id,years_exp
0,1,Alice,25,70000.0,10,3.0


In [ ]:
# --- isin() — filter by a list of values ---
print("Employees in dept 10 or 20:")
display(df[df["dept_id"].isin([10, 20])])

Employees in dept 10 or 20:


,emp_id,name,age,salary,dept_id,years_exp
0,1,Alice,25,70000.0,10,3.0
1,2,Beth,30,90000.0,20,7.0
2,3,Carol,22,65000.0,10,NaN
4,5,Eve,28,NaN,20,5.0


In [ ]:
# --- between() — filter a numeric range (inclusive) ---
print("Employees aged between 25 and 30:")
display(df[df["age"].between(25, 30)])

Employees aged between 25 and 30:


,emp_id,name,age,salary,dept_id,years_exp
0,1,Alice,25,70000.0,10,3.0
1,2,Beth,30,90000.0,20,7.0
4,5,Eve,28,NaN,20,5.0


In [ ]:
# --- query() — cleaner syntax for complex filters ---
print("Using query():")
display(df.query("age > 24 and dept_id == 10"))

Using query():


,emp_id,name,age,salary,dept_id,years_exp
0,1,Alice,25,70000.0,10,3.0


<a id='5'></a>
---
## 5. Handling Missing Data

Real-world datasets almost always have missing values (`NaN`). Pandas makes it easy to detect, remove, or fill them.

> 💡 **Tip:** Before deciding how to handle nulls, check their percentage per column. A column with >50% nulls may not be worth keeping at all.

In [ ]:
# --- Detecting missing values ---
print("Null count per column:")
print(df.isnull().sum())

print("\nNull percentage per column:")
print((df.isnull().sum() / len(df) * 100).round(1))

Null count per column:
emp_id       0
name         0
age          0
salary       1
dept_id      0
years_exp    1
dtype: int64

Null percentage per column:
emp_id        0.0
name          0.0
age           0.0
salary       20.0
dept_id       0.0
years_exp    20.0
dtype: float64


In [ ]:
# --- Filling missing values ---
df_clean = df.copy()  # always work on a copy!

# Fill numeric nulls with column mean
df_clean["salary"] = df_clean["salary"].fillna(df_clean["salary"].mean())

# Fill with a fixed value
df_clean["years_exp"] = df_clean["years_exp"].fillna(0)

print("After filling nulls:")
display(df_clean)

After filling nulls:


,emp_id,name,age,salary,dept_id,years_exp
0,1,Alice,25,70000.0,10,3.0
1,2,Beth,30,90000.0,20,7.0
2,3,Carol,22,65000.0,10,0.0
3,4,Dana,35,120000.0,30,9.0
4,5,Eve,28,86250.0,20,5.0


In [ ]:
# --- Dropping missing values ---
print("Drop rows with ANY null:")
display(df.dropna())

print("\nDrop rows where 'salary' is null only:")
display(df.dropna(subset=["salary"]))

Drop rows with ANY null:


,emp_id,name,age,salary,dept_id,years_exp
0,1,Alice,25,70000.0,10,3.0
1,2,Beth,30,90000.0,20,7.0
3,4,Dana,35,120000.0,30,9.0



Drop rows where 'salary' is null only:


,emp_id,name,age,salary,dept_id,years_exp
0,1,Alice,25,70000.0,10,3.0
1,2,Beth,30,90000.0,20,7.0
2,3,Carol,22,65000.0,10,NaN
3,4,Dana,35,120000.0,30,9.0


<a id='6'></a>
---
## 6. GroupBy

GroupBy splits data into groups, applies a function to each group, and combines the results — just like SQL's `GROUP BY`.

> 💡 **Key idea:** Think of it as: **Split → Apply → Combine**.

> 💡 **Tip:** Always call `reset_index()` after `groupby` to flatten the result back into a normal DataFrame.

In [ ]:
# --- Basic groupby ---
print("Average salary per department:")
display(df_clean.groupby("dept_id")["salary"].mean().reset_index())

Average salary per department:


,dept_id,salary
0,10,67500.0
1,20,88125.0
2,30,120000.0


In [ ]:
# --- Multiple aggregations with agg() ---
print("Multiple stats per department:")
display(
    df_clean.groupby("dept_id")["salary"]
    .agg(["mean", "min", "max", "count"])
    .reset_index()
)

Multiple stats per department:


,dept_id,mean,min,max,count
0,10,67500.0,65000.0,70000.0,2
1,20,88125.0,86250.0,90000.0,2
2,30,120000.0,120000.0,120000.0,1


In [ ]:
# --- Different aggregations per column ---
print("Mean salary and max age per department:")
display(
    df_clean.groupby("dept_id").agg({
        "salary": "mean",
        "age":    "max"
    }).reset_index()
)

Mean salary and max age per department:


,dept_id,salary,age
0,10,67500.0,25
1,20,88125.0,30
2,30,120000.0,35


In [ ]:
# --- Group by multiple columns ---
print("Average salary by dept and seniority (years_exp >= 7):")
df_clean["senior"] = df_clean["years_exp"] >= 7
display(
    df_clean.groupby(["dept_id", "senior"])["salary"]
    .mean()
    .reset_index()
)

Average salary by dept and seniority (years_exp >= 7):


,dept_id,senior,salary
0,10,False,67500.0
1,20,False,86250.0
2,20,True,90000.0
3,30,True,120000.0


<a id='7'></a>
---
## 7. Merging DataFrames

Pandas merge works like SQL JOINs. Use `merge()` to combine tables on a common key, and `concat()` to stack them.

| `how=` | Keeps |
|--------|-------|
| `"inner"` | Only matching rows in **both** |
| `"left"` | All rows from the **left** DataFrame |
| `"right"` | All rows from the **right** DataFrame |
| `"outer"` | All rows from **both** DataFrames |

In [ ]:
# Setup — two related tables
departments = pd.DataFrame({
    "dept_id":   [10, 20, 30],
    "dept_name": ["Engineering", "Marketing", "Finance"]
})

display(departments)

,dept_id,dept_name
0,10,Engineering
1,20,Marketing
2,30,Finance


In [ ]:
# --- Inner join (default) ---
print("Inner join — only matching rows:")
display(df_clean.merge(departments, on="dept_id", how="inner"))

Inner join — only matching rows:


,emp_id,name,age,salary,dept_id,years_exp,senior,dept_name
0,1,Alice,25,70000.0,10,3.0,False,Engineering
1,2,Beth,30,90000.0,20,7.0,True,Marketing
2,3,Carol,22,65000.0,10,0.0,False,Engineering
3,4,Dana,35,120000.0,30,9.0,True,Finance
4,5,Eve,28,86250.0,20,5.0,False,Marketing


In [ ]:
# --- Left join ---
# To properly demonstrate left join, we add an employee with no matching dept_id
extra = pd.DataFrame([{
    "emp_id": 99, "name": "Zara", "age": 29, "salary": 75000,
    "dept_id": 99, "years_exp": 4.0, "senior": False
}])
df_with_extra = pd.concat([df_clean, extra], ignore_index=True)

print("Left join — all employees kept; unmatched dept_id gets NaN for dept_name:")
display(df_with_extra.merge(departments, on="dept_id", how="left"))

Left join — all employees kept; unmatched dept_id gets NaN for dept_name:


,emp_id,name,age,salary,dept_id,years_exp,senior,dept_name
0,1,Alice,25,70000.0,10,3.0,False,Engineering
1,2,Beth,30,90000.0,20,7.0,True,Marketing
2,3,Carol,22,65000.0,10,0.0,False,Engineering
3,4,Dana,35,120000.0,30,9.0,True,Finance
4,5,Eve,28,86250.0,20,5.0,False,Marketing
5,99,Zara,29,75000.0,99,4.0,False,NaN


In [ ]:
# --- concat() — stack DataFrames vertically ---
new_employees = pd.DataFrame({
    "emp_id":    [6, 7],
    "name":      ["Frank", "Grace"],
    "age":       [27, 31],
    "salary":    [80000, 95000],
    "dept_id":   [10, 30],
    "years_exp": [4.0, 8.0],
    "senior":    [False, True]
})

combined = pd.concat([df_clean, new_employees], ignore_index=True)
print("Combined DataFrame:")
display(combined)

Combined DataFrame:


,emp_id,name,age,salary,dept_id,years_exp,senior
0,1,Alice,25,70000.0,10,3.0,False
1,2,Beth,30,90000.0,20,7.0,True
2,3,Carol,22,65000.0,10,0.0,False
3,4,Dana,35,120000.0,30,9.0,True
4,5,Eve,28,86250.0,20,5.0,False
5,6,Frank,27,80000.0,10,4.0,False
6,7,Grace,31,95000.0,30,8.0,True


<a id='8'></a>
---
## 8. Creating New Columns & `apply()`

> 💡 **Key idea:** For simple math, use direct assignment. For custom logic per row, use `apply()`. For value substitution from a dictionary, use `map()`.

In [ ]:
df2 = df_clean.copy()

# --- Direct assignment (simple math) ---
df2["bonus"] = df2["salary"] * 0.10
df2["salary_k"] = (df2["salary"] / 1000).round(1)

display(df2[["name", "salary", "bonus", "salary_k"]])

,name,salary,bonus,salary_k
0,Alice,70000.0,7000.0,70.0
1,Beth,90000.0,9000.0,90.0
2,Carol,65000.0,6500.0,65.0
3,Dana,120000.0,12000.0,120.0
4,Eve,86250.0,8625.0,86.2


In [ ]:
# --- apply() with a custom function ---
# Note: salary > 90000 for High, so exactly 90000 falls into Mid
def salary_band(salary):
    if salary > 90000:
        return "High"
    elif salary > 70000:
        return "Mid"
    else:
        return "Low"

df2["band"] = df2["salary"].apply(salary_band)
display(df2[["name", "salary", "band"]])

,name,salary,band
0,Alice,70000.0,Low
1,Beth,90000.0,Mid
2,Carol,65000.0,Low
3,Dana,120000.0,High
4,Eve,86250.0,Mid


In [ ]:
# --- apply() with a lambda (inline function) ---
df2["salary_eur"] = df2["salary"].apply(lambda x: round(x * 0.92, 2))
display(df2[["name", "salary", "salary_eur"]])

,name,salary,salary_eur
0,Alice,70000.0,64400.0
1,Beth,90000.0,82800.0
2,Carol,65000.0,59800.0
3,Dana,120000.0,110400.0
4,Eve,86250.0,79350.0


In [ ]:
# --- map() — replace values using a dictionary ---
df2["dept_name"] = df2["dept_id"].map({10: "Engineering", 20: "Marketing", 30: "Finance"})
display(df2[["name", "dept_id", "dept_name"]])

,name,dept_id,dept_name
0,Alice,10,Engineering
1,Beth,20,Marketing
2,Carol,10,Engineering
3,Dana,30,Finance
4,Eve,20,Marketing


<a id='9'></a>
---
## 9. Sorting

> 💡 **Tip:** Use `nlargest()` / `nsmallest()` instead of `sort_values` + `head()` when you just need the top or bottom N rows — it's more concise and readable.

In [ ]:
# --- sort_values() ---
print("Sorted by salary descending:")
display(df_clean.sort_values("salary", ascending=False))

Sorted by salary descending:


,emp_id,name,age,salary,dept_id,years_exp,senior
3,4,Dana,35,120000.0,30,9.0,True
1,2,Beth,30,90000.0,20,7.0,True
4,5,Eve,28,86250.0,20,5.0,False
0,1,Alice,25,70000.0,10,3.0,False
2,3,Carol,22,65000.0,10,0.0,False


In [ ]:
# --- Sort by multiple columns ---
print("Sort by dept_id asc, then salary desc:")
display(df_clean.sort_values(["dept_id", "salary"], ascending=[True, False]))

Sort by dept_id asc, then salary desc:


,emp_id,name,age,salary,dept_id,years_exp,senior
0,1,Alice,25,70000.0,10,3.0,False
2,3,Carol,22,65000.0,10,0.0,False
1,2,Beth,30,90000.0,20,7.0,True
4,5,Eve,28,86250.0,20,5.0,False
3,4,Dana,35,120000.0,30,9.0,True


In [ ]:
# --- nlargest() / nsmallest() ---
print("Top 3 highest salaries:")
display(df_clean.nlargest(3, "salary"))

print("\nBottom 2 salaries:")
display(df_clean.nsmallest(2, "salary"))

Top 3 highest salaries:


,emp_id,name,age,salary,dept_id,years_exp,senior
3,4,Dana,35,120000.0,30,9.0,True
1,2,Beth,30,90000.0,20,7.0,True
4,5,Eve,28,86250.0,20,5.0,False



Bottom 2 salaries:


,emp_id,name,age,salary,dept_id,years_exp,senior
2,3,Carol,22,65000.0,10,0.0,False
0,1,Alice,25,70000.0,10,3.0,False


<a id='10'></a>
---
## 10. Duplicates

> 💡 **Tip:** Always check for duplicates early in your analysis — duplicate rows silently inflate counts and aggregations.

In [ ]:
# Create a DataFrame with duplicates for demonstration
df_dupes = pd.concat([df_clean, df_clean.iloc[[0, 1]]], ignore_index=True)

print("Total rows (with duplicates):", len(df_dupes))
print("Number of duplicate rows:", df_dupes.duplicated().sum())

Total rows (with duplicates): 7
Number of duplicate rows: 2


In [ ]:
# --- View duplicate rows ---
print("Duplicate rows:")
display(df_dupes[df_dupes.duplicated()])

Duplicate rows:


,emp_id,name,age,salary,dept_id,years_exp,senior
5,1,Alice,25,70000.0,10,3.0,False
6,2,Beth,30,90000.0,20,7.0,True


In [ ]:
# --- Remove duplicates ---
df_no_dupes = df_dupes.drop_duplicates()
print("After removing duplicates:", len(df_no_dupes), "rows")

# Remove duplicates based on a specific column, keep last occurrence
df_no_dupes2 = df_dupes.drop_duplicates(subset=["name"], keep="last")
print("After deduplication by name (keep last):", len(df_no_dupes2), "rows")

After removing duplicates: 5 rows
After deduplication by name (keep last): 5 rows


In [ ]:
# --- Unique values ---
print("Unique dept_ids:", df_clean["dept_id"].unique())
print("Number of unique depts:", df_clean["dept_id"].nunique())

print("\nFrequency of each dept_id:")
print(df_clean["dept_id"].value_counts())

Unique dept_ids: [10 20 30]
Number of unique depts: 3

Frequency of each dept_id:
dept_id
10    2
20    2
30    1
Name: count, dtype: int64


<a id='11'></a>
---
## 11. Pivot Tables

Pivot tables reshape data into a wide, spreadsheet-like format — ideal for summarizing and presenting results.

| Parameter | Purpose |
|-----------|--------|
| `values` | Column to aggregate |
| `index` | Rows |
| `columns` | Columns |
| `aggfunc` | How to aggregate (mean, sum, count…) |
| `fill_value` | Replace NaN in output |

> 💡 **groupby vs pivot_table:** Use `groupby` for calculations and pipelines; use `pivot_table` when you want a clean, readable cross-tabulation for reports or presentations.

In [ ]:
# --- Basic pivot table ---
print("Average salary by dept and seniority:")
display(
    df_clean.pivot_table(
        values="salary",
        index="dept_id",
        columns="senior",
        aggfunc="mean",
        fill_value=0
    )
)

Average salary by dept and seniority:


senior,False,True
dept_id,,
10,67500.0,0.0
20,86250.0,90000.0
30,0.0,120000.0


In [ ]:
# --- Multiple aggregation functions ---
print("Mean and max salary by department:")
display(
    df_clean.pivot_table(
        values="salary",
        index="dept_id",
        aggfunc=["mean", "max", "count"]
    )
)

Mean and max salary by department:


,mean,max,count
,salary,salary,salary
dept_id,,,
10,67500.0,70000.0,2
20,88125.0,90000.0,2
30,120000.0,120000.0,1


In [ ]:
# --- Adding row/column totals with margins ---
print("With totals row and column:")
display(
    df_clean.pivot_table(
        values="salary",
        index="dept_id",
        columns="senior",
        aggfunc="mean",
        fill_value=0,
        margins=True,
        margins_name="Total"
    )
)

With totals row and column:


senior,False,True,Total
dept_id,,,
10,67500.0,0.0,67500.0
20,86250.0,90000.0,88125.0
30,0.0,120000.0,120000.0
Total,73750.0,105000.0,86250.0


<a id='12'></a>
---
## 12. ⚡ Quick Reference Cheatsheet

### 📦 Setup
```python
import pandas as pd
import numpy as np
```

### 📥 Loading Data
```python
pd.read_csv("file.csv")        # CSV
pd.read_excel("file.xlsx")     # Excel
pd.read_json("file.json")      # JSON
```

### 🔍 Exploring
```python
df.head()          # first 5 rows
df.tail()          # last 5 rows
df.shape           # (rows, columns)
df.info()          # types, nulls
df.describe()      # summary stats
df.columns         # column names
df.dtypes          # data types
```

### 🎯 Selecting
```python
df["col"]                  # single column → Series
df[["col1", "col2"]]       # multiple columns → DataFrame
df.loc[row, "col"]         # by label (inclusive slice)
df.iloc[row, col_num]      # by position (exclusive slice)
```

### 🔎 Filtering
```python
df[df["col"] > value]                          # single condition
df[(df["col1"] > x) & (df["col2"] == y)]       # AND
df[(df["col1"] > x) | (df["col2"] == y)]       # OR
df[df["col"].isin([a, b])]                     # isin
df[df["col"].between(low, high)]               # between
df.query("col1 > x and col2 == y")             # query syntax
```

### 🧹 Missing Data
```python
df.isnull().sum()                              # null count per column
df["col"] = df["col"].fillna(df["col"].mean()) # fill with mean
df["col"] = df["col"].fillna(0)               # fill with value
df.dropna()                                    # drop rows with any null
df.dropna(subset=["col"])                      # drop by specific column
```

### 📊 GroupBy
```python
df.groupby("col")["val"].mean()                        # single aggregation
df.groupby("col")["val"].agg(["mean","min","max"])     # multiple
df.groupby(["col1","col2"]).agg({"val": "mean"})       # multi-column group
.reset_index()                                          # always flatten after!
```

### 🔗 Merging
```python
df1.merge(df2, on="key", how="inner")   # inner join
df1.merge(df2, on="key", how="left")    # left join
df1.merge(df2, on="key", how="outer")   # outer join
pd.concat([df1, df2], ignore_index=True) # stack vertically
```

### ✏️ Creating Columns
```python
df["new"] = df["col"] * 2                          # direct assignment
df["new"] = df["col"].apply(my_function)           # custom function
df["new"] = df["col"].apply(lambda x: x * 2)      # lambda
df["new"] = df["col"].map({a: "x", b: "y"})       # dictionary map
```

### 🔃 Sorting
```python
df.sort_values("col", ascending=False)             # descending
df.sort_values(["col1","col2"], ascending=[True, False])  # multi-column
df.nlargest(n, "col")                              # top N rows
df.nsmallest(n, "col")                             # bottom N rows
```

### 🧽 Duplicates
```python
df.duplicated().sum()                              # count duplicates
df.drop_duplicates()                               # remove all
df.drop_duplicates(subset=["col"], keep="last")   # by column
df["col"].unique()                                 # unique values
df["col"].nunique()                                # count unique
df["col"].value_counts()                           # frequency table
```

### 📋 Pivot Tables
```python
df.pivot_table(
    values="val",        # what to aggregate
    index="row_col",     # rows
    columns="col_col",   # columns
    aggfunc="mean",      # mean / sum / count / max
    fill_value=0,        # replace NaN in output
    margins=True         # add totals row/column
)
```